# Stage 9 - Structural Break Detection (AFML Ch. 17)

SADF/GSADF tests for bubble episodes in NVDA log-price series.

**ADF model:** delta_y = alpha + beta*y_lag + sum(gamma*delta_y_lags) + eps
**H0:** beta=0  **H1:** beta>0 (explosive/bubble)

**Outputs:** P22_sadf_structural_breaks.png, nvda_bsadf.parquet

In [ ]:
# ============================================================
# TICKER PARAMETER — change this to run for any stock
# ============================================================
TICKER = 'NVDA'   # Options: AAPL AMZN BAC GOOGL JNJ JPM MSFT NVDA UNH XOM

import os, sys
sys.path.insert(0, os.path.abspath('../..'))

DATA_RAW     = f'../../data/raw/{TICKER}_raw.csv'
DATA_CLEAN   = f'../../data/processed/per_stock/{TICKER}_clean.parquet'
DATA_CUSUM   = f'../../data/processed/per_stock/{TICKER}_cusum_events.parquet'
DATA_BARS    = f'../../data/processed/per_stock/{TICKER}_dollar_bars.parquet'
DATA_LABELS  = f'../../data/processed/per_stock/{TICKER}_labels.parquet'
DATA_WEIGHTS = f'../../data/processed/per_stock/{TICKER}_weights.parquet'
DATA_FRACDIFF= f'../../data/processed/per_stock/{TICKER}_fracdiff.parquet'
DATA_FEATURES= f'../../data/processed/per_stock/{TICKER}_ts_features.parquet'
DATA_BSADF   = f'../../data/processed/per_stock/{TICKER}_bsadf.parquet'
FIG_DIR      = f'../../reports/figures/per_stock/{TICKER}'
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs('../../data/processed/per_stock', exist_ok=True)


In [ ]:
import sys, warnings, time
sys.path.insert(0, os.path.abspath('../..'))
import os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.structural_breaks import get_bsadf, get_gsadf, cv_sadf

FIGURES = FIG_DIR
plt.rcParams.update({'font.size': 11, 'grid.alpha': 0.3})
print('Imports OK')

## 1. Load data

In [ ]:
clean = pd.read_parquet(f'../../data/processed/per_stock/{TICKER}_clean.parquet')
log_p = np.log(clean['Adj Close'])
print(f'Series: {len(log_p)} obs  ({log_p.index[0].date()} to {log_p.index[-1].date()})')

## 2. SADF path (expanding window ADF)

min_sl=63 trading days (one quarter), lags=1. Expected runtime: 30-120s.

In [ ]:
MIN_SL, LAGS = 63, 1
t0 = time.perf_counter()
bsadf = get_bsadf(log_p, min_sl=MIN_SL, lags=LAGS)
elapsed = time.perf_counter() - t0
sadf_stat = float(bsadf.max())
sadf_date = bsadf.idxmax()
print(f'Done in {elapsed:.1f}s')
print(f'SADF statistic = {sadf_stat:.4f} at {sadf_date.date()}')
print(f'Non-NaN: {bsadf.notna().sum()}')

## 3. Monte Carlo critical values (200 reps, n=500)

In [ ]:
t0 = time.perf_counter()
cvs = cv_sadf(n=500, min_sl=MIN_SL, lags=LAGS, reps=200, seed=42)
elapsed = time.perf_counter() - t0
cv90, cv95, cv99 = cvs['90%'], cvs['95%'], cvs['99%']
print(f'Monte Carlo ({elapsed:.1f}s):')
for k, v in cvs.items(): print(f'  {k}: {v:.4f}')
print(f'SADF={sadf_stat:.4f} > 95% CV ({cv95:.4f}): {sadf_stat > cv95}')

## 4. Identify bubble episodes (contiguous periods above 95% CV)

In [ ]:
above        = bsadf > cv95
bubble_start = above & ~above.shift(1, fill_value=False)
bubble_end   = ~above & above.shift(1, fill_value=False)
starts = bsadf[bubble_start].index.tolist()
ends   = bsadf[bubble_end].index.tolist()
episodes = []
for s in starts:
    later = [e for e in ends if e > s]
    episodes.append((s, later[0] if later else bsadf.index[-1]))
print(f'{len(episodes)} episode(s) above 95% CV:')
for s, e in episodes:
    pk = bsadf[s:e].max()
    print(f'  {s.date()} to {e.date()}  peak={pk:.2f}  ({(e-s).days} days)')

## 5. P22 - SADF path overlaid on NVDA price (dual-axis)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), dpi=120, sharex=True,
                               gridspec_kw={'height_ratios': [2, 1]})
# Price
ax1.semilogy(clean.index, clean['Adj Close'], color='steelblue', lw=0.7, label='NVDA Adj Close')
for i, (s, e) in enumerate(episodes):
    ax1.axvspan(s, e, color='tomato', alpha=0.18, label='Bubble (>95% CV)' if i==0 else '_')
ax1.set_ylabel('Adj Close (log scale)')
ax1.set_title('NVDA SADF Structural Break Detection  (AFML Ch. 17)', fontweight='bold')
ax1.legend(fontsize=10); ax1.grid(True)
# SADF
valid = bsadf.dropna()
ax2.plot(valid.index, valid.values, color='darkorange', lw=0.7, label='BSADF')
ax2.axhline(cv99, color='darkred', ls=':', lw=1.2, label=f'99% CV ({cv99:.2f})')
ax2.axhline(cv95, color='red',     ls='--', lw=1.2, label=f'95% CV ({cv95:.2f})')
ax2.axhline(cv90, color='orange',  ls='-.', lw=1.0, label=f'90% CV ({cv90:.2f})')
ax2.axhline(0, color='grey', lw=0.5)
ax2.fill_between(valid.index, valid.values, cv95,
                 where=(valid.values > cv95), color='red', alpha=0.15)
for s, e in episodes: ax2.axvspan(s, e, color='tomato', alpha=0.18)
ax2.set_ylabel('BSADF t-statistic'); ax2.set_xlabel('Date')
ax2.legend(fontsize=10); ax2.grid(True)
ax2.xaxis.set_major_locator(mdates.YearLocator(2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.savefig(f'{FIGURES}/P22_sadf_structural_breaks.png', dpi=150, bbox_inches='tight')
plt.show(); print('Saved P22_sadf_structural_breaks.png')

## 6. GSADF on monthly series (O(T^2) feasible at ~240 obs)

In [ ]:
log_p_m = np.log(clean['Adj Close'].resample('ME').last())
print(f'Monthly: {len(log_p_m)} obs')
t0 = time.perf_counter()
gsadf_stat = get_gsadf(log_p_m, min_sl=4, lags=1)
bsadf_m    = get_bsadf(log_p_m, min_sl=4, lags=1)
elapsed = time.perf_counter() - t0
print(f'Done in {elapsed:.1f}s')
print(f'SADF  (monthly) = {float(bsadf_m.max()):.4f}')
print(f'GSADF (monthly) = {gsadf_stat:.4f}  (GSADF >= SADF by construction)')

## 7. Save outputs

In [ ]:
bsadf.to_frame('bsadf').to_parquet(f'../../data/processed/per_stock/{TICKER}_bsadf.parquet')
print(f"Saved {TICKER} data to {DATA_CLEAN}")
summary = pd.Series({'SADF_daily': round(sadf_stat,4), 'GSADF_monthly': round(gsadf_stat,4),
    'CV_90pct': round(cv90,4), 'CV_95pct': round(cv95,4), 'CV_99pct': round(cv99,4),
    'bubble_episodes': len(episodes), 'min_sl_days': MIN_SL})
print(summary.to_string())